# 006 [실습 프로젝트]

### 실습 목표

1. 아래 샘플 문서들을 벡터 저장소에 저장하는 코드를 작성합니다. 
   - 적절한 벡터 저장소 선택 
   - 임베딩 모델 설정
   - 문서 구조 설계 (metadata 정의)

2. 벡터 저장소를 사용하여 다음 기능을 구현합니다. 
   - 새로운 문서 추가
   - 문서 삭제
   - 문서 검색: 유사도 점수 계산, 메타데이터 기반 필터링 등 

### 실습 단계

- **1단계: 벡터 저장소 초기화**
   - Chroma, FAISS, Pinecone 중 하나를 선택하고 이유를 설명하세요
   -> Chroma 선택 간편하고, 무료고, 로컬에 바로 둘 수 있습니다. 소규모 프로젝트로 연습용으로 좋아서 선택.

- **2단계: 문서 저장**
   - 제공된 샘플 문서를 Document 객체로 변환하세요
   - metadata 구조를 설계하세요 (type, author 포함)

- **3단계: 문서 관리**
   - 새로운 문서 1개를 추가하세요
   - 특정 문서 1개를 삭제하세요

- **4단계: 문서 검색 구현**
   - [ ] 기본 유사도 검색
   - [ ] 메타데이터 필터링
   - [ ] 점수 포함 검색

`(1) Env 환경변수`

In [ ]:
import os
import warnings

# Tokenizers 병렬 처리 경고 억제
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 경고 억제 (선택사항)
warnings.filterwarnings('ignore', category=UserWarning)

from dotenv import load_dotenv
load_dotenv()

`(2) 기본 라이브러리`

In [ ]:
import os
from glob import glob

In [1]:
# 샘플 문서 데이터 
documents = [
    {"content": "인공지능 기술의 발전과 미래", "type": "article", "author": "김철수"},
    {"content": "데이터 분석 입문 가이드", "type": "tutorial", "author": "이영희"},
    {"content": "머신러닝 모델 성능 개선 방법", "type": "research", "author": "박지성"},
    {"content": "블록체인 기술과 금융 혁신", "type": "article", "author": "정민우"},
    {"content": "클라우드 컴퓨팅 아키텍처 설계", "type": "tutorial", "author": "강다은"},
    {"content": "사이버 보안 위협 대응 전략", "type": "research", "author": "홍길동"},
    {"content": "빅데이터 처리 시스템 구축 사례", "type": "article", "author": "송지원"},
    {"content": "웹 개발자를 위한 REST API 가이드", "type": "tutorial", "author": "임성준"},
    {"content": "자연어 처리 알고리즘 비교 연구", "type": "research", "author": "최유진"},
    {"content": "디지털 트랜스포메이션 성공 전략", "type": "article", "author": "백승호"},
    {"content": "파이썬으로 시작하는 데이터 시각화", "type": "tutorial", "author": "유미란"},
    {"content": "강화학습을 활용한 게임 AI 개발", "type": "research", "author": "조현우"},
    {"content": "5G 네트워크 기술 동향", "type": "article", "author": "윤서연"},
    {"content": "도커 컨테이너 실전 가이드", "type": "tutorial", "author": "장민석"},
    {"content": "추천 시스템 최적화 연구", "type": "research", "author": "신영수"},
    {"content": "스마트 시티 구현 기술", "type": "article", "author": "권태영"},
    {"content": "깃허브 활용 협업 가이드", "type": "tutorial", "author": "오지훈"},
    {"content": "컴퓨터 비전 응용 사례 연구", "type": "research", "author": "남궁민"},
    {"content": "양자 컴퓨팅의 현재와 미래", "type": "article", "author": "하은주"},
    {"content": "리액트 네이티브 앱 개발 입문", "type": "tutorial", "author": "문동현"},
    {"content": "음성인식 시스템 성능 평가", "type": "research", "author": "심준호"},
    {"content": "메타버스 플랫폼 개발 동향", "type": "article", "author": "류아린"},
    {"content": "NoSQL 데이터베이스 설계 패턴", "type": "tutorial", "author": "반승현"},
    {"content": "엣지 컴퓨팅 적용 사례 연구", "type": "research", "author": "주민정"},
    {"content": "디지털 헬스케어 기술 혁신", "type": "article", "author": "구본우"},
    {"content": "마이크로서비스 아키텍처 구현", "type": "tutorial", "author": "염지현"},
    {"content": "강화학습 기반 로봇 제어 연구", "type": "research", "author": "탁현우"},
    {"content": "친환경 IT 인프라 구축 방안", "type": "article", "author": "방승미"},
    {"content": "프론트엔드 성능 최적화 기법", "type": "tutorial", "author": "곽준영"},
    {"content": "시계열 데이터 예측 모델 연구", "type": "research", "author": "추민서"}
]

In [ ]:
# 2 Document 객체 생성
from langchain_core.documents import Document

doc_objects = []

for i, doc in enumerate(documents): 
    doc_obj = Document(
        page_content=doc["content"],
        metadata={"type": doc["type"], "author": doc["author"]},
    )
    doc_objects.append(doc_obj)

# uuid 생성
import uuid

doc_ids = [str(uuid.uuid4()) for _ in range(len(doc_objects))]


In [ ]:
# 1. 벡터 저장소에 문서를 저장할 때 적용할 임베딩 모델
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings_model = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

# 디비생성
chroma_db = Chroma(
    collection_name = 'practice_rag',
    embedding_function=embeddings_model,
    persist_directory='./chroma_db',
)

# 벡터 저장소에 저장 
#Chroma 간편하고, 무료고, 로컬에 바로 둘 수 있습니다. 소규모 프로젝트로 연습용으로 좋아서 선택.
docs_add=chroma_db.add_documents(documents=doc_objects, ids=doc_ids)

print(f"{len(docs_add)}개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.")
print(docs_add)

chroma_db.get

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 42602.23it/s]


30개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.
['7ba4fdc6-e7bc-40c2-945b-fd7a2e210378', '619c38df-0c27-4e31-96db-3da18dfd3dc7', 'e897ae63-6dad-42a1-87b0-fd79e29f47b6', '905423a4-23c3-4466-9749-3b19ab6be461', 'e01098b5-a8f2-44db-9a7e-15e7bdf6e29a', 'e475bbf4-bc10-45ba-ac8d-54c4eb8a61c4', 'f1d29c7c-edac-4ed3-a5ae-3a9dd9afeb01', '4f09c7bf-55ac-4de7-9ce2-73aa9589c1f6', 'c80e728d-76f5-421c-955c-2d3d99c05a79', '13fad665-4ce1-452a-ad3e-67c082de99c7', '71f0c59c-2d35-4b48-8efb-41afc5cd9489', '274d46ad-7f90-4d19-b347-cf5e7d0496cf', 'b77843e9-47c6-48fc-a090-9d1716d8bd7c', '8df0e06b-2669-4439-911a-36accb765e7c', '9e377fc2-4957-4d19-a13f-3f2ba71ce2ad', '0c5924c2-dd80-495b-a4ae-aa089b367fe5', 'e1089f14-8dbd-4144-8e09-4656e031fbdf', '6319614c-40bb-4b6c-be89-d5c338868a8b', '4e15a9bf-191c-4876-bf9f-e6c7d1cb2fc2', '4ae6a06a-357f-4bf8-9eb1-80560810b463', '0deb3ac0-3452-4c57-9a93-613a2c42b63d', '9f8b1bb2-46c9-4e43-ad7d-70b6f006e7af', '98a3f1b1-7e5e-4d9d-9288-22662816dbf2', '8a8d1196-41a8-4ffd-94d1-4bcc7b57c422', 'f31c42

<bound method Chroma.get of <langchain_chroma.vectorstores.Chroma object at 0x0000020F8011B290>>

In [ ]:
import chromadb

# 1. 크로마 클라이언트 연결 (예시: 로컬 저장소)
client = chromadb.PersistentClient(path="./chroma_db")

# 2. 기존 컬렉션 가져오기
collection = client.get_collection(name="practice_rag")

# 3. 데이터 개수 확인
data_count = collection.count()
print(f"현재 컬렉션에 저장된 데이터 개수: {data_count}개")

현재 컬렉션에 저장된 데이터 개수: 31개


In [ ]:
#3. 문서 추가

add_document = Document (
     page_content="추가데이터",
        metadata={"type": "research", "author":"조혜승"},
)
add_single_doc = chroma_db.add_documents(documents=[add_document], ids=[str(uuid.uuid4())])

print(f"{len(add_single_doc)}개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.")


1개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.


In [21]:
chroma_db.get()

{'ids': ['7ba4fdc6-e7bc-40c2-945b-fd7a2e210378',
  '619c38df-0c27-4e31-96db-3da18dfd3dc7',
  'e897ae63-6dad-42a1-87b0-fd79e29f47b6',
  '905423a4-23c3-4466-9749-3b19ab6be461',
  'e01098b5-a8f2-44db-9a7e-15e7bdf6e29a',
  'e475bbf4-bc10-45ba-ac8d-54c4eb8a61c4',
  'f1d29c7c-edac-4ed3-a5ae-3a9dd9afeb01',
  '4f09c7bf-55ac-4de7-9ce2-73aa9589c1f6',
  'c80e728d-76f5-421c-955c-2d3d99c05a79',
  '13fad665-4ce1-452a-ad3e-67c082de99c7',
  '71f0c59c-2d35-4b48-8efb-41afc5cd9489',
  '274d46ad-7f90-4d19-b347-cf5e7d0496cf',
  'b77843e9-47c6-48fc-a090-9d1716d8bd7c',
  '8df0e06b-2669-4439-911a-36accb765e7c',
  '9e377fc2-4957-4d19-a13f-3f2ba71ce2ad',
  '0c5924c2-dd80-495b-a4ae-aa089b367fe5',
  'e1089f14-8dbd-4144-8e09-4656e031fbdf',
  '6319614c-40bb-4b6c-be89-d5c338868a8b',
  '4e15a9bf-191c-4876-bf9f-e6c7d1cb2fc2',
  '4ae6a06a-357f-4bf8-9eb1-80560810b463',
  '0deb3ac0-3452-4c57-9a93-613a2c42b63d',
  '9f8b1bb2-46c9-4e43-ad7d-70b6f006e7af',
  '98a3f1b1-7e5e-4d9d-9288-22662816dbf2',
  '8a8d1196-41a8-4ffd-94d1-

In [ ]:
#3. 문서 삭제
result = chroma_db.get(where={"author": "조혜승"})

chroma_db.delete(ids=result['ids'])

In [ ]:
# 4. 유사도, 메타데이터필터
query ='AI나 머신러닝을 활용해서 성능을 높이거나 개발을 하는 방법에 대한 자료가 있나요?'
similarity_results = chroma_db.similarity_search(
    query,
    k=5,
    filter={'type':'tutorial'}
)
for doc in similarity_results:
    print(f"{doc.page_content} type: {doc.metadata['type']}")

# 점수포함
similarity_score_results = chroma_db.similarity_search_with_score(
    query,
    k=5,
    filter={'type':'tutorial'}
)   
  
for doc, score in similarity_score_results:
   
    print(f"{doc.page_content} type: {doc.metadata['type']} score: {score:.4f}")
    


프론트엔드 성능 최적화 기법 type: tutorial
웹 개발자를 위한 REST API 가이드 type: tutorial
깃허브 활용 협업 가이드 type: tutorial
데이터 분석 입문 가이드 type: tutorial
마이크로서비스 아키텍처 구현 type: tutorial
프론트엔드 성능 최적화 기법 type: tutorial score: 0.7567
웹 개발자를 위한 REST API 가이드 type: tutorial score: 0.9431
깃허브 활용 협업 가이드 type: tutorial score: 0.9892
데이터 분석 입문 가이드 type: tutorial score: 0.9975
마이크로서비스 아키텍처 구현 type: tutorial score: 1.0310


# 007 [실습 프로젝트] Naive RAG 구현 

- 각 단계별 지시사항에 따라 코드를 완성하세요. 
- 제시된 지시사항과 LangChain 문서를 참조하여 시스템을 구성합니다. 

`(1) 벡터 저장소 설정` 
- HuggingFace에서 지원하는 BAAI/bge-m3 임베딩 모델을 사용하여 문서를 벡터화
- FAISS DB를 벡터 스토어로 사용 (IndexFlatL2 사용: 유클리드 거리)

In [104]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings  
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
# Hugging Face의 임베딩 모델 생성
# 힌트: HuggingFaceEmbeddings(model_name="BAAI/bge-m3") 사용
embeddings_model = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

# 임베딩 차원 확인
embedding = embeddings_model.embed_query("test")
print(f"임베딩 차원: {len(embedding)}")

# 토크나이저 직접 접근
tokenizer = embeddings_model._client.tokenizer

# 토크나이저를 사용한 예시
text = "테스트 텍스트입니다."
tokens = tokenizer(text)
print(tokens)

# 토크나이저 설정 확인
print(tokenizer.model_max_length)  # 최대 토큰 길이
print(tokenizer.vocab_size) 

# 토큰 수를 계산하는 함수
def count_tokens(text):
    return len(tokenizer(text)['input_ids'])

# 토큰 수 계산
text = "테스트 텍스트입니다."
print(count_tokens(text))


# PDF 로더 초기화
pdf_loader = PyPDFLoader('./data/transformer.pdf')

# 동기 로딩
pdf_docs = pdf_loader.load()
print(f'PDF 문서 개수: {len(pdf_docs)}')
# 텍스트 분할기 생성
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,                      
    chunk_overlap=100,           
    length_function=count_tokens,         # 토큰 수를 기준으로 분할
    separators=["\n\n", "\n",],   # 구분자 - 재귀적으로 순차적으로 적용 
)

# 문서 id 생성
pdf_doc_ids = [str(uuid.uuid4()) for _ in range(len(chunks))]

# 텍스트 분할
chunks = text_splitter.split_documents(pdf_docs)
print(f"생성된 텍스트 청크 수: {len(chunks)}")
print(f"각 청크의 길이: {list(len(chunk.page_content) for chunk in chunks)}")
print(f"각 청크의 토큰 수: {list(count_tokens(chunk.page_content) for chunk in chunks)}")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 37217.98it/s]


임베딩 차원: 1024
{'input_ids': [0, 153924, 239355, 5826, 5, 2], 'attention_mask': [1, 1, 1, 1, 1, 1]}
8192
250002
6
PDF 문서 개수: 15
생성된 텍스트 청크 수: 38
각 청크의 길이: [1378, 1796, 1831, 1857, 1292, 1609, 503, 1555, 1278, 1365, 1608, 833, 1416, 1679, 999, 1764, 1604, 539, 1219, 1645, 926, 1213, 1688, 716, 1409, 1626, 624, 1411, 1438, 914, 1496, 1340, 847, 812, 470, 438, 470, 441]
각 청크의 토큰 수: [336, 415, 405, 419, 327, 424, 127, 389, 294, 382, 412, 205, 419, 417, 226, 419, 395, 149, 390, 400, 221, 356, 411, 181, 394, 405, 188, 424, 400, 278, 423, 413, 252, 178, 128, 115, 128, 111]


In [105]:
# Hugging Face 임베딩 모델을 사용한 FAISS 벡터 저장소 생성
import faiss 
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

# FAISS 인덱스 초기화 (유클리드 거리 사용)
dim = 1024  # 임베딩 차원
faiss_index = faiss.IndexFlatL2(dim)  

# FAISS 벡터 저장소 생성 및 transformer.pdf 문서를 벡터 저장소에 저장
faiss_db = FAISS.from_documents(
    documents=chunks, 
    embedding=embeddings_model, 
    ids=pdf_doc_ids
)

# 저장된 문서의 갯수 확인
print(faiss_db.index.ntotal)

38


In [ ]:
import uuid

# PDF 로더 초기화
pdf_loader = PyPDFLoader('./data/invest.pdf') #한눈에보는 상상인 자산전략.PDF

# 동기 로딩
pdf_docs = pdf_loader.load()
print(f'PDF 문서 개수: {len(pdf_docs)}')
# 텍스트 분할기 생성
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,                      
    chunk_overlap=100,           
    length_function=count_tokens,         # 토큰 수를 기준으로 분할
    separators=["\n\n", "\n",],   # 구분자 - 재귀적으로 순차적으로 적용 
)

new_doc = text_splitter.split_documents(pdf_docs)
# 문서 id 생성
doc_ids = [str(uuid.uuid4()) for _ in range(len(new_doc))]

# 문서를 벡터 저장소에 저장
# 힌트: faiss_db.add_documents(chunks, ids=doc_ids) 사용
added_doc_ids = faiss_db.add_documents(new_doc, ids=doc_ids)

# 벡터 저장소에 저장된 문서를 확인
print(f"{len(added_doc_ids)}개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.")
print(added_doc_ids)    

PDF 문서 개수: 23
43개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.
['71466368-a228-49a7-ac4d-72f31927fe71', 'c619d69f-489f-4a02-b642-d654265a458c', '707ee911-3097-4de7-9b64-87568c63919a', 'fed8a7f4-e61e-48f9-b3cc-ae2845a14bef', '5d8e2791-2118-40b5-b77e-d362262f57c1', 'abb5bd9d-6e49-435b-8723-e0f09db398b3', '2a898b50-9b87-42b5-89e9-809489c8e531', 'be05fd9f-74dd-4e0b-8bda-9d226ac5919e', '054b04fd-a983-4f49-b296-355880a95b37', '50845b2b-ab65-4c28-9afc-79a20b28ed35', '11a49c72-7608-4536-8534-ec554b129ddd', '03b32f53-62b0-46ad-bc15-8b5959182310', '703dc901-afbf-4dc9-8db3-3c8c7983c3e3', '48e8883c-9d54-46a6-91dc-90ef5adce6d5', 'd8a504dd-a64a-4fd9-b380-6d1d81394052', '050b1ea5-a0a1-4151-9d97-0e6660823231', '90236030-e342-4e12-a902-df1acb7fd8f8', '9d46ca34-4d96-48ca-a3a0-8957d5ff4584', '3fc9bd59-8add-4398-90fd-383175924331', 'f4e6d4a7-29d7-4c91-b0a4-b834b093ccd8', 'a9bc9127-617e-4bb8-864e-cd19e1f8b41d', '8e951081-38fa-4eeb-8dda-c7a99d3fd131', '316d37f4-d97e-424c-a189-7146dee03398', 'fa45f095-46c3-42d1-84e2-45255264

In [107]:
print(faiss_db.index.ntotal)

81


`(2) 검색기 정의`
- mmr 검색으로 상위 3개 문서 검색하는 Retriever 사용
- 다양성을 높이는 설정을 사용 

In [108]:
# Faiss DB를 기반으로 MMR 검색기 생성
# lambda_mult를 0.3으로 낮춰서 서로 비슷한 내용보다는 다양한 주제의 문서가 나오도록 유도합니다.

faiss_mmr_retriever = faiss_db.as_retriever(
    search_type='mmr', 
    search_kwargs={'k': 3, 'fetch_k': 10, 'lambda_mult': 0.3}
)

# 잘 생성되었는지 확인 (테스트)
query = "스페이스X상장 이후 변동성"
retrieved_docs = faiss_mmr_retriever.invoke(query)

print(f"=== MMR 검색 결과 (총 {len(retrieved_docs)}개 문서) ===")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"-{i}-\n{doc.page_content[:100]}...{doc.page_content[-100:]}")
    print("-" * 100) 


=== MMR 검색 결과 (총 3개 문서) ===
-1-
엑시트 가능성이 높다는 점에 시장 인식이 모아질 수 있겠다. 이를 보수적으로 해석한다면, 
급격한 "Risk ON" 보다는 완연한 "변동성 축소" 국면으로의 전환에 진입할 가능성이... 악재에 
대한 적응력이 높아지고 있다는 점이 중요하겠다. 시장 주도권은 지정학적 리스크와 유가에서 
점차 유동성 그리고 금리와 직결되는 연준 통화정책으로 이동할 가능성이 높겠다.
----------------------------------------------------------------------------------------------------
-2-
30
50
70
90
110
130
26.01 26.03 26.05
WTI 원유선물
코스피 (우)
(USD/bbl, 역축) (pt)
4,000
5,000
6,000
7,000
8,...
7,900
26.01 26.03 26.05
s&p500 코스피 (우)(pt) (pt)
4/22 이후  
WTI-코스피 
디커플링 
4월이후 코스피  
S&P500과 
유사한 흐름
----------------------------------------------------------------------------------------------------
-3-
6/12(금): SpaceX 나스닥 상장 
현지시각 6월 12일 SpaceX(티커 SPCX)가 나스닥에 상장하며, 차주는 상장 후 첫 한 주의 가격 
발견이 진행된다. 공모가는 주...물가 안정으로 직결된다. 
반대로 관세와 무역 불균형을 둘러싼 이견이 부각될 경우 공급망 물가 우려가 재차 고개를 들 
수 있어, 회의 전후 유가와 달러의 반응을 함께 봐야 한다.
----------------------------------------------------------------------------------------------------


`(3) RAG 프롬프트 구성`

- 작성 기준: 
    - LangChain의 ChatPromptTemplate 클래스 사용
    - 변수 처리는 {context}, {question} 형식 사용
    - 답변은 한글로 출력되도록 프롬프트 작성
    
- 아래 템플릿 코드를 기반으로 다음 내용을 참고하여 작성합니다. 

    1. 프롬프트 구성요소:
        - 작업 지침
        - 컨텍스트 영역
        - 질문 영역
        - 답변 형식 가이드

    2. 작업 지침:
        - 컨텍스트 기반 답변 원칙
        - 외부 지식 사용 제한
        - 불확실성 처리 방법
        - 답변 불가능한 경우의 처리 방법

    3. 답변 형식:
        - 핵심 답변 섹션
        - 근거 제시 섹션
        - 추가 설명 섹션 (필요시)

    4. 제약사항 반영:
        - 답변은 사실에 기반해야 함
        - 추측이나 가정을 최소화해야 함
        - 명확한 근거 제시가 필요함
        - 구조화된 형태로 작성되어야 함

In [109]:
# Prompt 템플릿 (여기에 작성하세요)
from langchain_core.prompts import ChatPromptTemplate

system_template ="""당신은 주어진 컨텍스트(Context) 정보만을 바탕으로 질문에 답변하는 친절하고 정확한 AI 어시스턴트입니다.

[작업 지침 및 제약사항]
1. 반드시 아래 제공된 [Context]의 정보만을 기반으로 답변을 작성하세요.
2. 절대 당신이 기존에 알고 있는 외부 지식이나 상식을 활용하여 답변하지 마세요.
3. [Context] 내용에 기반한 사실만을 서술해야 하며, 절대 추측, 가정, 상상을 더하지 마세요.
4. 만약 질문에 대한 명확한 답을 [Context]에서 찾을 수 없거나 불확실하다면, 억지로 답변을 꾸며내지 말고 반드시 다음 문장만 정확하게 출력하세요: "제공된 컨텍스트에서 관련 정보를 찾을 수 없어 답변이 불가능합니다."
5. 모든 답변은 가독성이 좋게 구조화된 형태로 작성되어야 하며, 반드시 한국어(한글)로 출력해야 합니다.

[답변 형식 가이드]
출력할 때는 반드시 아래의 3가지 섹션 구조를 엄격히 지켜서 작성하세요:

■ 핵심 답변
- 질문에 대한 명확하고 간결한 핵심 결론을 서술합니다.

■ 근거 제시
- [Context]의 어떤 문장이나 문맥을 바탕으로 위 답변이 도출되었는지 사실에 기반한 명확한 근거를 서술합니다.

■ 추가 설명 (선택사항)
- 핵심 답변을 보완하기 위해 [Context] 내에 존재하는 추가 정보가 있다면 기술합니다. (없다면 이 섹션은 제외하거나 생략합니다.)

[Context]
{context}
"""

# 2. 질문 영역 정의
human_template = """  [질문]
{question}"""

# 3. ChatPromptTemplate 생성 (시스템 메시지와 휴먼 메시지 결합)
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("human", human_template)
])

# 4. 프롬프트 구성 확인 테스트 (실제 사용 예시)
print("=== 프롬프트 구성 완료 ===")

prompt = ChatPromptTemplate.from_template(system_template)

# 템플릿 출력
prompt.pretty_print()

=== 프롬프트 구성 완료 ===
================================ Human Message =================================

당신은 주어진 컨텍스트(Context) 정보만을 바탕으로 질문에 답변하는 친절하고 정확한 AI 어시스턴트입니다.

[작업 지침 및 제약사항]
1. 반드시 아래 제공된 [Context]의 정보만을 기반으로 답변을 작성하세요.
2. 절대 당신이 기존에 알고 있는 외부 지식이나 상식을 활용하여 답변하지 마세요.
3. [Context] 내용에 기반한 사실만을 서술해야 하며, 절대 추측, 가정, 상상을 더하지 마세요.
4. 만약 질문에 대한 명확한 답을 [Context]에서 찾을 수 없거나 불확실하다면, 억지로 답변을 꾸며내지 말고 반드시 다음 문장만 정확하게 출력하세요: "제공된 컨텍스트에서 관련 정보를 찾을 수 없어 답변이 불가능합니다."
5. 모든 답변은 가독성이 좋게 구조화된 형태로 작성되어야 하며, 반드시 한국어(한글)로 출력해야 합니다.

[답변 형식 가이드]
출력할 때는 반드시 아래의 3가지 섹션 구조를 엄격히 지켜서 작성하세요:

■ 핵심 답변
- 질문에 대한 명확하고 간결한 핵심 결론을 서술합니다.

■ 근거 제시
- [Context]의 어떤 문장이나 문맥을 바탕으로 위 답변이 도출되었는지 사실에 기반한 명확한 근거를 서술합니다.

■ 추가 설명 (선택사항)
- 핵심 답변을 보완하기 위해 [Context] 내에 존재하는 추가 정보가 있다면 기술합니다. (없다면 이 섹션은 제외하거나 생략합니다.)

[Context]
{context}



`(4) RAG 체인 구성`
- LangChain의 LCEL 문법을 사용
- 검색 결과를 프롬프트의 'context'로 전달하고,
- 사용자가 입력한 질문을 그래도 프롬프트의 'question'에 전달
- LLM 설정:
    - ChatOpenAI 사용 ('gpt-4o-mini' 모델)
    - temperature: 답변의 일관성을 가져가는 설정값을 사용 
    - 기타 필요한 설정 
- 출력 파서: 문자열 부분만 출력되도록 구성

In [115]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# LLM 설정
# 힌트: ChatOpenAI(model='gpt-4o-mini', temperature=0) 사용
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
# 문서 포맷팅
def format_docs(new_doc):
    return "\n\n".join([f"{doc.page_content}" for doc in new_doc])

# RAG 체인 생성
# 힌트: {'context': faiss_mmr_retriever | format_docs, 'question': RunnablePassthrough()} | prompt | llm | StrOutputParser()
rag_chain ={'context': faiss_mmr_retriever | format_docs, 'question': RunnablePassthrough()} | prompt | llm | StrOutputParser()

# 체인 실행
query = "국내 증시 차주 전망"
output = rag_chain.invoke(query)

print(f"쿼리: {query}")
print("답변:")
print(output)

쿼리: 국내 증시 차주 전망
답변:
■ 핵심 답변
- 증시는 선진국과 신흥국 간의 대조적인 흐름을 보이고 있으며, 미국 주식은 상대적으로 견조한 흐름을 나타내고 있습니다.

■ 근거 제시
- "증시는 선진국과 신흥국의 대조적 흐름이 극명히 나타났다. 미국 주식은 주간 기준 +0.12% 상승하며 주요 자산 가운데 상대적으로 견조한 흐름을 나타냈다."라는 문장을 통해 미국 주식의 상승세와 견조함을 확인할 수 있습니다.

■ 추가 설명
- 반면, 신흥국은 -3.08%, 중국은 -2.77%의 큰 폭의 조정을 기록하며 미국과의 수익률 격차가 확대되었다고 언급되어 있습니다. 이는 신흥국의 투자심리 회복이 제한되고 있음을 나타냅니다.


`(5) Gradio 스트리밍 구현`
- ChatInterface 사용
- `chain.stream()`으로 응답을 청크 단위로 스트리밍

In [116]:
import gradio as gr
from typing import Iterator

# 스트리밍 응답 생성 함수
def get_streaming_response(message: str, history) -> Iterator[str]:
    
    response = ""
    
    # 핵심 수정: message를 명확하게 문자열로 고정하여 stream에 전달합니다.
    # history는 사용하지 않더라도 매개변수(history) 자리는 그대로 유지해야 Gradio가 에러를 내지 않습니다.
    for chunk in rag_chain.stream(str(message)):
        if isinstance(new_doc, str):
            response += new_doc
            yield response

# Gradio 인터페이스 설정
demo = gr.ChatInterface(
    fn=get_streaming_response, 
    title="🤖 RAG 기반 질의응답 시스템", 
    description="문서 내용을 바탕으로 답변하는 스트리밍 챗봇입니다.",
    examples=["스페이스X 상장 이후 현황", "미국 5월 CPI리뷰 및 향후 물가 전망","국내 증시 차주 전망"]
)

# 실행 (만약 colab이나 로컬 환경에 따라 share=True를 추가할 수도 있습니다)
demo.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


In [117]:
# 인터페이스 종료
demo.close()

Closing server running on port: 7866
